Applying 3x3 filter on spatial image of size 5x5

this reduces size of image = to retain same size of image we can do:
1. zero padding <br>
0 0 0 0 0 0 <br>
0 5 4 6 4 0 <br>
0 3 3 4 5 0 <br>
0 3 5 4 2 0 <br>
0 0 0 0 0 0 <br>

2. pixel border replication <br>
```
_ _ _ _ _ _
_ 5 4 6 4 _
```

3. Wrap-around padding
4.

weighted filter h(x,y) = max(|x|,|y|)

size of kernel always taken odd

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def create_averaging_filter(size):
  kernel = np.ones((size,size),dtype=np.float32)
  kernel = kernel / np.sum(kernel)
  return kernel

In [ ]:
def create_weighted_filter(size):
  center = size // 2
  kernel = np.zeros((size,size), dtype=np.float32)

  for i in range(size):
    for j in range(size):
      x = i-center
      y = j-center

      value = max(abs(x), abs(y))
      kernel[i][j] = value

  total = 0
  for i in range(size):
    for j in range(size):
      total = total + kernel[i][j]

  if total == 0:
    total = 1   #avoid div by zero when size = 1

  for i in range(size):
    for j in range(size):
      kernel[i][j] = kernel[i][j] / total

  return kernel

In [ ]:
def create_gaussian_filter(size, sigma):
  center = size // 2
  kernel = np.zeros((size,size), dtype=np.float32)

  for i in range(size):
    for j in range(size):
      x = i-center
      y = j-center

      value = np.exp(-(x*x + y*y) / (2*sigma*sigma))
      kernel[i][j] = value

  total = 0
  for i in range(size):
    for j in range(size):
      total = total + kernel[i][j]

  for i in range(size):
    for j in range(size):
      kernel[i][j] = kernel[i][j] / total

  return kernel

padding functions:

1. cv2.border_constant = zero padding
2. cv2.border_replicate
3. cv2.border_reflect

In [ ]:
def do_padding(image, pad, padding_type):

  height = image.shape[0]
  width = image.shape[1]

  padded_height = height + 2*pad
  padded_width = width + 2*pad

  padded_image = np.zeros((padded_height, padded_width), dtype=np.float32)

  for i in range(height):
    for j in range(width):
      padded_image[i+pad][j+pad] = image[i][j]

  # padding_type 1 = zero padding -> nothing more to do, borders stay 0

  if padding_type == 2:
    for i in range(pad):
      for j in range(width):
        padded_image[i][j+pad] = image[0][j]
        padded_image[padded_height-1-i][j+pad] = image[height-1][j]

    for j in range(pad):
      for i in range(padded_height):
        padded_image[i][j] = padded_image[i][pad]
        padded_image[i][padded_width-1-j] = padded_image[i][padded_width-1-pad]

  if padding_type == 3: 
    for i in range(pad):
      for j in range(width):
        padded_image[pad-1-i][j+pad] = image[i][j]
        padded_image[padded_height-pad+i][j+pad] = image[height-1-i][j]

    for j in range(pad):
      for i in range(padded_height):
        padded_image[i][pad-1-j] = padded_image[i][pad+j]
        padded_image[i][padded_width-pad+j] = padded_image[i][padded_width-pad-1-j]

  return padded_image

In [ ]:
def apply_filter(image, kernel, padding_type):

  size = kernel.shape[0]   #kernel is size x size
  pad = size // 2

  height = image.shape[0]
  width = image.shape[1]

  padded_image = do_padding(image, pad, padding_type)

  output_image = np.zeros((height, width), dtype=np.float32)

  for i in range(height):
    for j in range(width):

      pixel_sum = 0
      for m in range(size):
        for n in range(size):
          pixel_sum = pixel_sum + padded_image[i+m][j+n] * kernel[m][n]

      output_image[i][j] = pixel_sum

  output_image = np.clip(output_image, 0, 255)
  output_image = output_image.astype(np.uint8)

  return output_image

In [ ]:
image = cv2.imread('images.jpg')
image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

plt.imshow(image_gray, cmap='gray')
plt.title('Original Image (Grayscale)')
plt.axis('off')
plt.show()

In [ ]:
print("Choose filter type:")
print("1. Averaging filter")
print("2. Weighted filter h(x,y) = max(|x|,|y|)")
print("3. Gaussian filter")
filter_choice = int(input("Enter choice (1/2/3): "))

filter_size = int(input("Enter filter size (odd number, e.g. 3, 5, 7): "))

sigma = 1.0
if filter_choice == 3:
  sigma = float(input("Enter sigma value for Gaussian filter (e.g. 1.0): "))

print("Choose padding type:")
print("1. Zero padding")
print("2. Replicate padding")
print("3. Reflect padding")
padding_choice = int(input("Enter choice (1/2/3): "))

In [ ]:
if filter_choice == 1:
  kernel = create_averaging_filter(filter_size)
  filter_name = "Averaging Filter"

elif filter_choice == 2:
  kernel = create_weighted_filter(filter_size)
  filter_name = "Weighted Filter (max(|x|,|y|))"

elif filter_choice == 3:
  kernel = create_gaussian_filter(filter_size, sigma)
  filter_name = "Gaussian Filter"

print(filter_name, "kernel:")
print(kernel)

In [ ]:
result = apply_filter(image_gray, kernel, padding_choice)

plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(image_gray, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(result, cmap='gray')
plt.title(filter_name + ' (size=' + str(filter_size) + ')')
plt.axis('off')

plt.show()